In [10]:
from requests import get
from re import findall as find

html = open("1.html", "r", encoding="utf-8").readlines()

shuliandu = "https://cbg-hp.res.netease.com/game_res/unpack_res/cbg/others/pic_shuliandu_huo_01.png"
kuang_pre = "https://cbg-hp.res.netease.com/game_res/unpack_res/cbg/others/pic_!!!_??.png"
num_pre = "https://cbg-hp.res.netease.com/game_res/unpack_res/cbg/others/pic_card_num_b!.png"

cards = {}

card_url = ""
kuang_url = ""
leixing = ""
pinzhi = ""
num = ""
name = ""
num_url = ""

pinzhi_map = {
            "01": "普通",
            "02": "稀有",
            "03": "史诗",
            "04": "传说",
            "05": "禁忌",
            "06": "光辉",
        }

for line in html:
    if 'class="cqm-text icon-label">' in line:
        name = find(r'class="cqm-text icon-label">(.+?)</span>', line)[0]
        if name == "赫敏·格兰杰" and pinzhi == "光辉":
            name = "小赫敏·格兰杰"
        if name == "哈利·波特" and pinzhi == "光辉":
            name = "小哈利·波特"
        if name in cards:
            print(name)
        cards[name] = {
            "魔力消耗": num,
            "类型": leixing,
            "品质": pinzhi,
            "卡片url": card_url,
            "框url": kuang_url,
            "魔力url": num_url
        }
        card_url = kuang_url = leixing = num = pinzhi = name = num_url = ""
    if 'skill_card_small' in line:
        card_url = find(r'(https://cbg-hp.res.netease.com/game_res/image/skill_card_small/.+?.png)', line)[0]
    if "kuang" in line:
        kuang_url = find(r'(https.+?.png)', line)[0]
        if "zhuchang" in kuang_url:
            leixing = "召唤卡"
        elif "huoban" in kuang_url:
            leixing = "伙伴卡"
        else:
            leixing = "咒语卡"
        pinzhi_num = find(r"[0-9]+", kuang_url)[0]
        pinzhi = pinzhi_map[pinzhi_num]
    if "pic_card_num_b" in line:
        num_url = find(r"https:.+?.png", line)[0]
        num = find(r"[0-9]+", num_url)[0]
        num = int(num)
        


In [11]:
import json
from pypinyin import lazy_pinyin

def sort_dict_by_pinyin(d):
    return dict(sorted(d.items(), key=lambda item: lazy_pinyin(item[0])))

sorted_data = sort_dict_by_pinyin(cards)
json.dump(sorted_data, open("cards.json", "w", encoding="utf-8"), ensure_ascii=False, indent=4)

In [12]:
import os
def download(url, name):
    if os.path.exists(name):
        return
    
    with open(name, "wb") as f:
        f.write(get(url).content)
        print(f"下载 {name} 完成")
    return name

In [13]:
download(shuliandu, "background/shuliandu.png")

for name, card in cards.items():
    num = card["魔力消耗"]
    leixing = card["类型"]
    pinzhi = card["品质"]
    card_url = card["卡片url"]
    kuang_url = card["框url"]
    num_url = card["魔力url"]
    
    download(kuang_url, f"background/{leixing}_{pinzhi}.png")
    if num_url != "":
        download(num_url, f"background/card_num_{num}.png")
    download(card_url, f"background/{name}.png")
    